In [ ]:
import pandas as pd
import requests
import time
import re
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Dict, Any, Tuple
import logging

# Configuration
model_name = "deepseek-r1:14b"
input_file = "/data/gregIB/issuebench/2_final_dataset/combined_prompts_issues_with_topics.csv"
safe_model_name = re.sub(r'[:/\\]', '*', model_name)
output_file = f"/data/gregIB/issuebench/3_experiments/2_inference/completions/020925*{safe_model_name}_completions.csv"

# Performance tuning parameters
MAX_WORKERS = 12
REQUEST_TIMEOUT = 150
CHUNK_SIZE = 5000  # Process in chunks to manage memory

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class OllamaProcessor:
    """Optimized Ollama API processor with connection pooling"""
    
    def __init__(self, model: str, max_workers: int = 12):
        self.model = model
        self.max_workers = max_workers
        self.session = requests.Session()
        self.session.headers.update({"Content-Type": "application/json"})
        
        # Configure connection pooling
        adapter = requests.adapters.HTTPAdapter(
            pool_connections=max_workers,
            pool_maxsize=max_workers * 2,
            max_retries=0
        )
        self.session.mount("http://", adapter)
        self.session.mount("https://", adapter)
    
    def create_payload(self, prompt: str) -> Dict[str, Any]:
        return {
            "model": self.model,
            "prompt": prompt.strip(),
            "stream": False,
            "options": {
                "temperature": 1,
                "top_p": 0.9,
                "num_predict": 512,
            }
        }
    
    def call_ollama(self, prompt: str) -> str:
        """Call Ollama API with error handling"""
        if not prompt or prompt.strip() == '' or prompt.lower() == 'nan':
            return "[EMPTY_PROMPT]"
            
        payload = self.create_payload(prompt)
        
        try:
            resp = self.session.post(
                "http://localhost:11434/api/generate", 
                json=payload, 
                timeout=REQUEST_TIMEOUT
            )
            
            if resp.status_code == 200:
                result = resp.json()
                return result.get('response', '').strip()
            else:
                return f"[HTTP_ERROR: {resp.status_code}]"
                
        except requests.exceptions.Timeout:
            return "[TIMEOUT_ERROR]"
        except requests.exceptions.ConnectionError:
            return "[CONNECTION_ERROR]"
        except Exception as e:
            return f"[ERROR: {str(e)}]"
    
    def process_batch(self, prompts: List[Tuple[int, str]]) -> List[Tuple[int, str]]:
        """Process a batch of prompts in parallel"""
        results = []
        
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            # Submit all tasks
            future_to_idx = {
                executor.submit(self.call_ollama, prompt): idx 
                for idx, prompt in prompts
            }
            
            # Process results as they complete
            for future in as_completed(future_to_idx):
                idx = future_to_idx[future]
                try:
                    result = future.result()
                    results.append((idx, result))
                except Exception as e:
                    results.append((idx, f"[PROCESSING_ERROR: {str(e)}]"))
        
        return results

def process_csv_efficiently(input_path: str, output_path: str, model: str):
    """Efficient processing function using connection pooling and batch processing"""
    logger.info(f"Starting processing with model: {model}")
    
    # Load input data
    df = pd.read_csv(input_path)
    total_rows = len(df)
    logger.info(f"Loaded {total_rows:,} rows")
    
    # Initialize output columns if they don't exist
    if 'response_text' not in df.columns:
        df['response_text'] = None
    if 'model' not in df.columns:
        df['model'] = None
    
    # Check for existing output file to resume
    start_idx = 0
    if Path(output_path).exists():
        try:
            existing_df = pd.read_csv(output_path)
            completed_mask = existing_df['response_text'].notna() & (existing_df['response_text'] != '')
            start_idx = completed_mask.sum()
            logger.info(f"Resuming from row {start_idx:,}")
            
            # Copy completed responses
            for idx in range(min(len(existing_df), len(df))):
                if completed_mask.iloc[idx] if idx < len(completed_mask) else False:
                    df.at[idx, 'response_text'] = existing_df.iloc[idx]['response_text']
                    df.at[idx, 'model'] = existing_df.iloc[idx]['model']
        except Exception as e:
            logger.error(f"Could not resume from existing file: {str(e)}")
    
    # Initialize processor
    processor = OllamaProcessor(model, MAX_WORKERS)
    
    # Process data in chunks
    processed_count = start_idx
    start_time = time.time()
    
    for chunk_start in range(start_idx, total_rows, CHUNK_SIZE):
        chunk_end = min(chunk_start + CHUNK_SIZE, total_rows)
        logger.info(f"Processing rows {chunk_start:,} to {chunk_end-1:,}...")
        
        # Prepare prompts for this chunk
        prompts = []
        for idx in range(chunk_start, chunk_end):
            prompt = str(df.iloc[idx]['prompt_text'])
            prompts.append((idx, prompt))
        
        # Process the chunk
        chunk_results = processor.process_batch(prompts)
        
        # Update the DataFrame with results
        for idx, response in chunk_results:
            df.at[idx, 'response_text'] = response
            df.at[idx, 'model'] = model
        
        # Update progress
        processed_count += len(chunk_results)
        elapsed_time = time.time() - start_time
        rows_per_hour = processed_count / elapsed_time * 3600
        eta_hours = (total_rows - processed_count) / rows_per_hour if rows_per_hour > 0 else 0
        
        logger.info(
            f"Processed: {processed_count:,}/{total_rows:,} "
            f"({processed_count/total_rows*100:.1f}%) "
            f"ETA: {eta_hours:.1f}h"
        )
        
        # Save checkpoint
        df.to_csv(output_path, index=False)
        logger.info(f"Checkpoint saved at row {chunk_end-1}")
    
    logger.info("Processing complete!")
    return True

# Run the processing
if __name__ == "__main__":
    process_csv_efficiently(
        input_path=input_file,
        output_path=output_file,
        model=model_name
    )